In [69]:
import sagemaker
import boto3
import os

sess = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

print("region:", region)
print("role:", role)

region: us-east-1
role: arn:aws:iam::110276528929:role/SageMakerStudioExecutionRole2026


In [70]:
# Definimos información de nuestra imagen Docker en AWS

# Aquí obtenemos el ID de nuestra cuenta de AWS.
# Esto es necesario porque las imágenes en ECR se identifican
# con el formato: account-id.dkr.ecr.region.amazonaws.com/repositorio:tag

# Por eso primero necesitamos saber cuál es nuestro account id.

account = boto3.client("sts").get_caller_identity()["Account"]


# Definimos el nombre del repositorio en ECR

# Este será el lugar donde guardaremos la imagen Docker 
# que contiene nuestro training container.

ecr_repository = "ml-training-byoc"


# Definimos el tag de la imagen

# En Docker cada imagen tiene un tag (versión).
# Aquí usamos "latest" para indicar la versión más reciente.


tag = "latest"


# Construimos el URI completo de la imagen

# Este URI será utilizado por SageMaker para encontrar
# la imagen Docker que ejecutará el training job.

# Formato:
# account.dkr.ecr.region.amazonaws.com/repository:tag

image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{ecr_repository}:{tag}"
print(image_uri)

110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-training-byoc:latest


In [71]:
# Creando repositorio en Amazon ECR

# ECR (Elastic Container Registry) es donde AWS guarda las imágenes Docker que luego SageMaker utilizará para
# ejecutar el training container.

# Aquí creamos un repositorio donde se almacenará
# la imagen Docker de nuestro modelo.


import boto3

ecr = boto3.client("ecr")

try:
    response = ecr.create_repository(repositoryName=ecr_repository)
    print("Repositorio creado:", response["repository"]["repositoryUri"])
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("El repositorio ya existe:", ecr_repository)

El repositorio ya existe: ml-training-byoc


In [60]:
# Creamos el Estimator de SageMaker

# Un Estimator es el objeto que SageMaker usa para
# ejecutar el entrenamiento del modelo.

# Aquí le indicamos a SageMaker:
# - qué imagen Docker usar (image_uri)
# - qué rol de AWS usar (role)
# - qué tipo de instancia usar para entrenar
# - cuántas instancias usar

from sagemaker.estimator import Estimator

estimator = Estimator(
    image_uri=image_uri, # imagen Docker en ECR
    role=role, # rol de SageMaker
    instance_count=1, # número de instancias
    instance_type="ml.m5.large", # instancia recomendada
    base_job_name="training-byoc",
    sagemaker_session=sess
)

print("Estimator creado correctamente")

Estimator creado correctamente


In [72]:
# Subimos los datos a S3 para que SageMaker pueda acceder a ellos

s3_data_path = sess.upload_data(
    path="../data/prep",
    key_prefix="training-data"
)

print("Datos subidos a:", s3_data_path)

Datos subidos a: s3://sagemaker-us-east-1-110276528929/training-data


In [74]:
# Lanzamos el training job en SageMaker

# Con estimator.fit() le decimos a SageMaker que ejecute
# el entrenamiento usando el container Docker que definimos.

# SageMaker hará lo siguiente automáticamente:
# 1. Crear una instancia EC2 para entrenamiento
# 2. Descargar la imagen Docker desde ECR
# 3. Ejecutar nuestro script de training dentro del container
# 4. Guardar el modelo entrenado en S3

estimator.fit({
    "train": s3_data_path
})

print("Training job lanzado")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: training-byoc-2026-03-08-06-03-10-446


2026-03-08 06:03:10 Starting - Starting the training job...
2026-03-08 06:03:33 Starting - Preparing the instances for training...
2026-03-08 06:03:58 Downloading - Downloading input data...
2026-03-08 06:04:33 Training - Training image download completed. Training in progress.Downloading ruff (10.6MiB)
 Downloaded ruff
Installed 8 packages in 10ms
2026-03-08 06:04:44,015 - __main__ - INFO - Iniciando entrenamiento.
2026-03-08 06:04:46,627 - __main__ - INFO - Datasets cargados. train_rows=7068600 valid_rows=214200
2026-03-08 06:04:46,630 - __main__ - INFO - Features=47 | cat_features=4
2026-03-08 06:04:46,630 - __main__ - INFO - Entrenando clasificador (venta vs no venta).
2026-03-08 06:11:37,612 - __main__ - INFO - Clasificador entrenado. prob_valid_mean=0.1109
2026-03-08 06:11:37,612 - __main__ - INFO - Entrenando regresor (unidades | venta).
2026-03-08 06:12:23,762 - __main__ - INFO - Regresor entrenado. mu_valid_mean=1.4657
2026-03-08 06:12:23,764 - __main__ - INFO - Alpha fijo usa

In [75]:
print(estimator.latest_training_job.name)
print(estimator.model_data)

training-byoc-2026-03-08-06-03-10-446
s3://sagemaker-us-east-1-110276528929/training-byoc-2026-03-08-06-03-10-446/output/model.tar.gz


In [81]:
# Creamos el modelo de SageMaker para inferencia en tiempo real

# Este objeto Model le dice a SageMaker:
# - qué imagen Docker usar para servir inferencias
# - dónde está el modelo entrenado en S3
# - qué rol de AWS usar para crear el endpoint

from sagemaker.model import Model
from time import strftime, gmtime

# Definimos el URI completo de la imagen de inferencia en ECR

# Esta imagen contiene nuestra lógica de serving
# y será utilizada por SageMaker al levantar el endpoint.

inference_image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/ml-inference-byoc:v2"

# model_data apunta al artifact generado por el training job,
# es decir, al model.tar.gz que SageMaker guardó en S3 automáticamente.

model_data = "s3://sagemaker-us-east-1-110276528929/training-byoc-2026-03-08-00-03-04-733/output/model.tar.gz"

model_name = f"ml-inference-byoc-model-{strftime('%Y%m%d-%H%M%S', gmtime())}"

model = Model(
    image_uri=inference_image_uri,
    model_data=model_data,
    role=role,
    sagemaker_session=sess,
    name=model_name,
)

print("Modelo SageMaker creado correctamente")
print("model_name:", model_name)
print("model_data:", model_data)
print("inference_image_uri:", inference_image_uri)

Modelo SageMaker creado correctamente
model_name: ml-inference-byoc-model-20260308-070523
model_data: s3://sagemaker-us-east-1-110276528929/training-byoc-2026-03-08-00-03-04-733/output/model.tar.gz
inference_image_uri: 110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-inference-byoc:v2


In [83]:
# Levantamos el endpoint en tiempo real con SageMaker

# Con model.deploy() le decimos a SageMaker que:
# 1. Cree una configuración de endpoint
# 2. Levante una instancia para serving
# 3. Descargue la imagen Docker desde ECR
# 4. Descargue el modelo entrenado desde S3
# 5. Exponga un endpoint para inferencias en tiempo real

from time import strftime, gmtime

endpoint_name = f"ml-inference-byoc-endpoint-{strftime('%Y%m%d-%H%M%S', gmtime())}"

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
)

print("Endpoint levantado correctamente")
print("endpoint_name:", endpoint_name)

INFO:sagemaker:Creating model with name: ml-inference-byoc-model-20260308-070523
INFO:sagemaker:Creating endpoint-config with name ml-inference-byoc-endpoint-20260308-071040
INFO:sagemaker:Creating endpoint with name ml-inference-byoc-endpoint-20260308-071040


-----!Endpoint levantado correctamente
endpoint_name: ml-inference-byoc-endpoint-20260308-071040


In [84]:
# Creamos manualmente el objeto Predictor para conectarnos al endpoint

# Este objeto Predictor nos permite enviar requests de inferencia
# al endpoint que SageMaker ya dejó activo.
# Usamos el endpoint_name que acabamos de crear.

from sagemaker.predictor import Predictor

predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess,
)

print("Predictor conectado correctamente")
print("endpoint_name:", endpoint_name)

Predictor conectado correctamente
endpoint_name: ml-inference-byoc-endpoint-20260308-071040


In [86]:
# Generamos una muestra válida de inferencias en tiempo real

# En lugar de inventar un payload incompleto,
# tomamos una fila real del dataset ya preprocesado.
# Así garantizamos que el endpoint reciba exactamente
# las mismas columnas que el modelo espera en producción.

import json
import pandas as pd

sample_df = pd.read_parquet("../data/prep/test_features.parquet").head(1)

sample_payload = sample_df.to_dict(orient="records")

response = predictor.predict(
    data=json.dumps(sample_payload),
    initial_args={"ContentType": "application/json"},
)

print("Respuesta del endpoint:")
print(response)

Respuesta del endpoint:
b'{"predictions": [0.0753730982542038]}'


In [87]:
# Decodificamos la respuesta del endpoint para inspeccionarla mejor

# SageMaker devuelve la respuesta en bytes.
# Aquí la convertimos a JSON para verla de forma legible.

response_json = json.loads(response.decode("utf-8"))

print("Predicciones:")
print(response_json)

Predicciones:
{'predictions': [0.0753730982542038]}
